# Expert-CFG: test.json 생성

**방법**: VQA-RAD 이미지 피처(pre-extracted) × PEIR 캡션 텍스트 피처(BiomedCLIP 인코딩) → cross-modal 검색

**Kaggle Dataset 구조:**
```
/kaggle/input/expert-cfg-data/
├── VQA-RAD/
│   ├── features/          # synpic*.h5 (이미지 피처, pre-extracted)
│   └── images/
├── PEIR_openai_imgid2caption.json
├── PEIR_openai_match_keyword.json
└── filter_cap.json
```

In [ ]:
!pip install open_clip_torch datasets h5py -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print("HF_TOKEN 설정 완료")

In [ ]:
import os

DATA_ROOT    = "/kaggle/input/expert-cfg-data"
VQA_RAD_ROOT = f"{DATA_ROOT}/VQA-RAD"
FEATURES_DIR = f"{VQA_RAD_ROOT}/features"
FILTER_CAP   = f"{DATA_ROOT}/filter_cap.json"
IMGID2CAP    = f"{DATA_ROOT}/PEIR_openai_imgid2caption.json"
MATCH_KW     = f"{DATA_ROOT}/PEIR_openai_match_keyword.json"
OUTPUT_PATH  = "/kaggle/working/test.json"
TOP_K        = 5

for path in [FEATURES_DIR, FILTER_CAP, IMGID2CAP, MATCH_KW]:
    print(f"{'✅' if os.path.exists(path) else '❌'} {path}")

## Step 1. BiomedCLIP 로드

In [ ]:
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model, _, _ = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
tokenizer = open_clip.get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model = model.to(device).eval()
print("BiomedCLIP 로드 완료")

## Step 2. PEIR 캡션 텍스트 피처 인코딩 (~15분)

In [ ]:
import json
import numpy as np
from tqdm.notebook import tqdm

# PEIR 캡션 로드
with open(IMGID2CAP) as f:
    imgid2cap = json.load(f)

peir_ids = sorted(imgid2cap.keys(), key=lambda x: int(x))
peir_captions = [imgid2cap[pid]["caption"] for pid in peir_ids]
peir_keywords = [imgid2cap[pid]["keywords"] for pid in peir_ids]
print(f"PEIR 이미지 수: {len(peir_ids)}")

# 배치 인코딩
BATCH_SIZE = 256
all_text_feats = []

with torch.no_grad():
    for i in tqdm(range(0, len(peir_captions), BATCH_SIZE), desc="인코딩"):
        batch = peir_captions[i:i+BATCH_SIZE]
        tokens = tokenizer(batch).to(device)
        feats = model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_text_feats.append(feats.cpu().float().numpy())

text_feat_matrix = np.concatenate(all_text_feats, axis=0)
print(f"텍스트 피처 matrix: {text_feat_matrix.shape}")

## Step 3. VQA-RAD 이미지 피처 로드

In [ ]:
import h5py

img_features = {}
h5_files = [f for f in os.listdir(FEATURES_DIR) if f.endswith(".h5")]

for fname in h5_files:
    img_id = fname.replace(".h5", ".jpg")
    with h5py.File(os.path.join(FEATURES_DIR, fname), "r") as f:
        feat = f["feature"][()].squeeze().astype(np.float32)
        feat = feat / (np.linalg.norm(feat) + 1e-8)
        img_features[img_id] = feat

print(f"VQA-RAD 이미지 피처: {len(img_features)}개")

## Step 4. match_keyword 로드

In [ ]:
with open(MATCH_KW) as f:
    match_keyword = json.load(f)

# PEIR id → match_keyword 역 인덱스
peir_id_to_highlights = {}
for cap_text, info in match_keyword.items():
    for pid in info.get("img_ids", []):
        pid_str = str(pid)
        if pid_str not in peir_id_to_highlights:
            peir_id_to_highlights[pid_str] = []
        peir_id_to_highlights[pid_str].extend(info.get("match_keyword", []))

print(f"highlight 인덱스: {len(peir_id_to_highlights)}개")

## Step 5. VQA-RAD 로드 + test.json 생성

In [ ]:
from datasets import load_dataset

ds = load_dataset("flaviagiammarino/vqa-rad", split="test")
print(f"VQA-RAD test: {len(ds)}개")
print(ds[0])

In [ ]:
annotations = []
skipped = 0

for item in tqdm(ds, desc="test.json 생성"):
    img_id = str(item.get("image_name") or item.get("img_id") or item.get("image_id", ""))
    if not img_id.endswith(".jpg"):
        img_id = img_id + ".jpg"

    base = {
        "img_id": img_id,
        "question": item["question"],
        "answer": item["answer"],
        "answer_type": item.get("answer_type", "OPEN"),
    }

    if img_id not in img_features:
        skipped += 1
        base["top_k_captions"] = [""]
        base["highlights"] = []
        annotations.append(base)
        continue

    # cosine similarity (이미지 피처는 이미 정규화됨)
    img_feat = img_features[img_id]
    sims = text_feat_matrix @ img_feat
    top_idx = np.argsort(sims)[::-1][:TOP_K]

    top_captions = []
    all_highlights = []

    for idx in top_idx:
        pid = peir_ids[idx]
        cap = peir_captions[idx]
        if cap:
            top_captions.append(cap)
        kws = peir_id_to_highlights.get(pid, [])
        all_highlights.extend(kws)

    base["top_k_captions"] = top_captions if top_captions else [""]
    base["highlights"] = list(dict.fromkeys(all_highlights))
    annotations.append(base)

print(f"완료: {len(annotations)}개, 피처 없음: {skipped}개")

with open(OUTPUT_PATH, "w") as f:
    json.dump(annotations, f, indent=2, ensure_ascii=False)
print(f"✅ 저장: {OUTPUT_PATH}")

## Step 6. 결과 확인

In [ ]:
sample = annotations[0]
print(f"img_id     : {sample['img_id']}")
print(f"question   : {sample['question']}")
print(f"answer     : {sample['answer']}")
print(f"top_caption: {sample['top_k_captions'][0][:120]}")
print(f"highlights : {sample['highlights'][:5]}")
print(f"\n✅ test.json → {OUTPUT_PATH}")